# Direct Cloud Access: Sentinel-3 Multidimensional NetCDF Cubes

**Objective:** A data engineering proof-of-concept demonstrating direct cloud-filesystem access to complex multidimensional NetCDF data cubes (Sentinel-3 OLCI) without local downloads.
*   **The Problem:** While modern Cloud Optimised GeoTIFFs (COGs) are easy to load, advanced oceanographic sensors such as Sentinel-3 store data in complex, multi-layered NetCDF formats. Traditional geospatial loaders often fail to parse these nested structures, and downloading massive global satellite swaths to a local machine is highly inefficient.
*   **The Solution:** This pipeline bypasses standard geospatial engines. It uses Python's native cloud-filesystem reader (`fsspec`) with `xarray` to open and manipulate multidimensional data cubes directly from the Microsoft Planetary Computer. It also demonstrates how to unpack complex 32-bit Water Quality and Science Flags (WQSF) to extract valid biogeochemical pixels.
*   **Key Libraries:** `xarray`, `fsspec`, `pystac-client`, `h5netcdf`
---
*   **Author:** Jade Farrugia | Marine Biogeochemical Modeller & PhD Candidate
*   **Contact:** https://www.linkedin.com/in/jadefarrugia/
*   **Date:** August 2026

In [1]:
import pystac_client
import planetary_computer
import xarray as xr
import fsspec
import numpy as np

# 1. Query the Wadjemup Continental Shelf
print("Querying Sentinel-3 over Wadjemup Shelf...")
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
bbox = [115.4, -32.1, 115.6, -31.9] # Rottnest bounds

search = catalog.search(
    collections=["sentinel-3-olci-wfr-l2-netcdf"],
    bbox=bbox,
    datetime="2023-01-01/2023-01-31"
)
items = list(search.items())
item = items[0] # Grab the first available image pass

# 2. Directly open the NetCDF files using xarray and fsspec
print("Loading NetCDF assets directly via xarray...")

# Open the Chlorophyll asset
chl_file = fsspec.open(item.assets["chl-nn"].href).open()
ds_chl = xr.open_dataset(chl_file, engine="h5netcdf")

# Open the Water Quality Flags asset
wqsf_file = fsspec.open(item.assets["wqsf"].href).open()
ds_wqsf = xr.open_dataset(wqsf_file, engine="h5netcdf")

# Extract the actual data variables from inside the NetCDF files
chl_data = ds_chl["CHL_NN"]
wqsf_data = ds_wqsf["WQSF"]

# 3. Print the available WQSF flag meanings so you can see the bitmask options
print("--- WQSF Flag Definitions ---")
if "flag_meanings" in ds_wqsf["WQSF"].attrs:
    print(ds_wqsf["WQSF"].attrs["flag_meanings"])
else:
    print("WQSF loaded successfully as 32-bit flag array.")

# 4. Check for valid Chlorophyll readings
# xarray automatically masks land and clouds as NaN
valid_chl = chl_data.where(~np.isnan(chl_data))

# 5. Calculate Usable Coverage
total_pixels = chl_data.size
valid_pixels = np.count_nonzero(~np.isnan(valid_chl.values))

coverage = (valid_pixels / total_pixels) * 100

print("\n-------------------------------------------")
print(f"Total Bounding Box Pixels: {total_pixels}")
print(f"Valid Biogeochemical Pixels: {valid_pixels}")
print(f"Usable Biogeochemical Coverage over Wadjemup Shelf: {coverage:.1f}%")
print("-------------------------------------------")

Querying Sentinel-3 over Wadjemup Shelf...
Loading NetCDF assets directly via xarray...
--- WQSF Flag Definitions ---
INVALID WATER LAND CLOUD TURBID_ATM CLOUD_AMBIGUOUS CLOUD_MARGIN SNOW_ICE INLAND_WATER COASTLINE TIDAL COSMETIC SUSPECT HISOLZEN SATURATED MEGLINT HIGHGLINT WHITECAPS ADJAC WV_FAIL PAR_FAIL AC_FAIL OC4ME_FAIL OCNN_FAIL KDM_FAIL BPAC_ON WHITE_SCATT LOWRW HIGHRW ANNOT_ANGSTROM ANNOT_AERO_B ANNOT_ABSO_D ANNOT_ACLIM ANNOT_ABSOA ANNOT_MIXR1 ANNOT_DROUT ANNOT_TAU06 RWNEG_O1 RWNEG_O2 RWNEG_O3 RWNEG_O4 RWNEG_O5 RWNEG_O6 RWNEG_O7 RWNEG_O8 RWNEG_O9 RWNEG_O10 RWNEG_O11 RWNEG_O12 RWNEG_O16 RWNEG_O17 RWNEG_O18 RWNEG_O21

-------------------------------------------
Total Bounding Box Pixels: 19902715
Valid Biogeochemical Pixels: 1476935
Usable Biogeochemical Coverage over Wadjemup Shelf: 7.4%
-------------------------------------------
